In [1]:
import os
import sys
import time

JAVA_HOME = r"C:\Users\alexandre.karneff\Java\jdk-17"

if JAVA_HOME and os.path.isdir(JAVA_HOME):
    os.environ["JAVA_HOME"] = JAVA_HOME
    os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ["PATH"]

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F



In [2]:
spark = SparkSession.builder.master("local[2]").appName("Mushroom_Classification").getOrCreate()
spark

c:\Users\alexandre.karneff\Documents\BD_PySpark\.envpyspark\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
sc = spark.sparkContext
sc

<SparkContext master=local[2] appName=Mushroom_Classification>

# 1. Charger les données

In [5]:
DATA_PATH = "data/mushrooms.csv"

df_raw = spark.read.csv(DATA_PATH, header=True, inferSchema=False)

print(f"{df_raw.count()} lignes, {len(df_raw.columns)} colonnes")
df_raw.printSchema()

8124 lignes, 23 colonnes
root
 |-- class: string (nullable = true)
 |-- cap-shape: string (nullable = true)
 |-- cap-surface: string (nullable = true)
 |-- cap-color: string (nullable = true)
 |-- bruises: string (nullable = true)
 |-- odor: string (nullable = true)
 |-- gill-attachment: string (nullable = true)
 |-- gill-spacing: string (nullable = true)
 |-- gill-size: string (nullable = true)
 |-- gill-color: string (nullable = true)
 |-- stalk-shape: string (nullable = true)
 |-- stalk-root: string (nullable = true)
 |-- stalk-surface-above-ring: string (nullable = true)
 |-- stalk-surface-below-ring: string (nullable = true)
 |-- stalk-color-above-ring: string (nullable = true)
 |-- stalk-color-below-ring: string (nullable = true)
 |-- veil-type: string (nullable = true)
 |-- veil-color: string (nullable = true)
 |-- ring-number: string (nullable = true)
 |-- ring-type: string (nullable = true)
 |-- spore-print-color: string (nullable = true)
 |-- population: string (nullable = tr

In [6]:
df = df_raw.toDF(*[c.replace("-", "_") for c in df_raw.columns])
df.show(5)
# Remplacer les tirets des noms des colonnes en underscore

+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|class|cap_shape|cap_surface|cap_color|bruises|odor|gill_attachment|gill_spacing|gill_size|gill_color|stalk_shape|stalk_root|stalk_surface_above_ring|stalk_surface_below_ring|stalk_color_above_ring|stalk_color_below_ring|veil_type|veil_color|ring_number|ring_type|spore_print_color|population|habitat|
+-----+---------+-----------+---------+-------+----+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+---------+----------+-----------+---------+-----------------+----------+-------+
|    p|        x|          s|        n|      t|   p|              f|           c|        n|   

In [7]:
# Répartition de la target
total = df.count()
(
    df.groupBy("class").count()
    .withColumn("pct", F.round(F.col("count") / total * 100, 1))
    .orderBy("class")
    .show()
)

+-----+-----+----+
|class|count| pct|
+-----+-----+----+
|    e| 4208|51.8|
|    p| 3916|48.2|
+-----+-----+----+



In [8]:
# Le dataset n'a pas de `null`, mais les valeurs manquantes sont codées par `?`
features_all = [c for c in df.columns if c != "class"]

df.select(
    [F.sum(F.when(F.col(c).isNull() | (F.col(c) == "?"), 1).otherwise(0)).alias(c) for c in df.columns]
).show(vertical=True)

-RECORD 0------------------------
 class                    | 0    
 cap_shape                | 0    
 cap_surface              | 0    
 cap_color                | 0    
 bruises                  | 0    
 odor                     | 0    
 gill_attachment          | 0    
 gill_spacing             | 0    
 gill_size                | 0    
 gill_color               | 0    
 stalk_shape              | 0    
 stalk_root               | 2480 
 stalk_surface_above_ring | 0    
 stalk_surface_below_ring | 0    
 stalk_color_above_ring   | 0    
 stalk_color_below_ring   | 0    
 veil_type                | 0    
 veil_color               | 0    
 ring_number              | 0    
 ring_type                | 0    
 spore_print_color        | 0    
 population               | 0    
 habitat                  | 0    



Seule stalk-root a des manquants mais environ 30%

In [9]:
n_distinct = df.agg(*[F.countDistinct(c).alias(c) for c in df.columns]).first().asDict()
for col, n in sorted(n_distinct.items(), key=lambda kv: kv[1]):
    print(f"{col:28s} {n}")

veil_type                    1
class                        2
bruises                      2
gill_attachment              2
gill_spacing                 2
gill_size                    2
stalk_shape                  2
ring_number                  3
cap_surface                  4
stalk_surface_above_ring     4
stalk_surface_below_ring     4
veil_color                   4
stalk_root                   5
ring_type                    5
cap_shape                    6
population                   6
habitat                      7
odor                         9
stalk_color_above_ring       9
stalk_color_below_ring       9
spore_print_color            9
cap_color                    10
gill_color                   12


veil_type n'a qu'une seule valeur, pas utile on l'enlèvera

In [10]:
df.crosstab("odor", "class").orderBy("odor_class").show()

+----------+----+----+
|odor_class|   e|   p|
+----------+----+----+
|         a| 400|   0|
|         c|   0| 192|
|         f|   0|2160|
|         l| 400|   0|
|         m|   0|  36|
|         n|3408| 120|
|         p|   0| 256|
|         s|   0| 576|
|         y|   0| 576|
+----------+----+----+



L'odeur sépare presque à elle seule les classes

In [11]:
# Même lecture pour la couleur de la sporée, autre variable très informative
df.crosstab("spore_print_color", "class").orderBy("spore_print_color_class").show()

+-----------------------+----+----+
|spore_print_color_class|   e|   p|
+-----------------------+----+----+
|                      b|  48|   0|
|                      h|  48|1584|
|                      k|1648| 224|
|                      n|1744| 224|
|                      o|  48|   0|
|                      r|   0|  72|
|                      u|  48|   0|
|                      w| 576|1812|
|                      y|  48|   0|
+-----------------------+----+----+



La couleur Spore est aussi très discriminante

## Nettoyer

- On supprime `veil_type` (constante).
- Pour `stalk_root`, on **garde** les `?` comme une modalité à part entière, renommée `missing` :
  le fait que la racine n'ait pas été observée peut lui-même être informatif, et supprimer 30 % des lignes serait du gâchis.
- On vérifie les doublons.

In [12]:
df_clean = (
    df.drop("veil_type")
      .withColumn("stalk_root", F.when(F.col("stalk_root") == "?", "missing").otherwise(F.col("stalk_root")))
)

print("Doublons :", df_clean.count() - df_clean.dropDuplicates().count())

feature_cols = [c for c in df_clean.columns if c != "class"]
print(len(feature_cols), "variables explicatives :", feature_cols)

Doublons : 0
21 variables explicatives : ['cap_shape', 'cap_surface', 'cap_color', 'bruises', 'odor', 'gill_attachment', 'gill_spacing', 'gill_size', 'gill_color', 'stalk_shape', 'stalk_root', 'stalk_surface_above_ring', 'stalk_surface_below_ring', 'stalk_color_above_ring', 'stalk_color_below_ring', 'veil_color', 'ring_number', 'ring_type', 'spore_print_color', 'population', 'habitat']


## Split

In [13]:
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.cache()
test_df = test_df.cache()

print(f"Train : {train_df.count()} lignes | Test : {test_df.count()} lignes")
train_df.groupBy("class").count().orderBy("class").show()
test_df.groupBy("class").count().orderBy("class").show()

Train : 6573 lignes | Test : 1551 lignes
+-----+-----+
|class|count|
+-----+-----+
|    e| 3420|
|    p| 3153|
+-----+-----+

+-----+-----+
|class|count|
+-----+-----+
|    e|  788|
|    p|  763|
+-----+-----+



## Preprocessing

Les modèles Spark ML attendent :
- une colonne **`label`** numérique ;
- une colonne **`features`** de type vecteur.

On enchaîne donc :

| Étape | Rôle |
|---|---|
| `StringIndexer` sur `class` | `e` → 0, `p` → 1 (ordre alphabétique : **1 = toxique = classe positive**) |
| `StringIndexer` sur les 21 variables | chaque lettre → un entier |
| `OneHotEncoder` | chaque entier → vecteur binaire (évite qu'un modèle linéaire croie que `x=3` > `x=1`) |
| `VectorAssembler` | regroupe tout dans `features` |

`handleInvalid="keep"` évite qu'une modalité absente du train (mais présente dans le test) fasse planter le pipeline.

In [14]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

In [15]:
def build_prep_stages(cols):
    """Étapes de préparation pour une liste de variables catégorielles."""
    idx_cols = [f"{c}_idx" for c in cols]
    ohe_cols = [f"{c}_ohe" for c in cols]
    return [
        StringIndexer(inputCol="class", outputCol="label", stringOrderType="alphabetAsc"),
        StringIndexer(inputCols=cols, outputCols=idx_cols, handleInvalid="keep"),
        OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols, handleInvalid="keep"),
        VectorAssembler(inputCols=ohe_cols, outputCol="features"),
    ]


In [16]:
prep_model = Pipeline(stages=build_prep_stages(feature_cols)).fit(train_df)

In [17]:
train_prep = prep_model.transform(train_df).select("label", "features").cache()
test_prep = prep_model.transform(test_df).select("label", "features").cache()


In [27]:
n_features = train_prep.first()["features"].size
print("Taille du vecteur features :", n_features)
train_prep.show(5, truncate=80)

Taille du vecteur features : 137
+-----+--------------------------------------------------------------------------------+
|label|                                                                        features|
+-----+--------------------------------------------------------------------------------+
|  0.0|(137,[3,9,13,23,26,36,40,42,49,59,62,68,73,77,87,97,103,106,112,125,130],[1.0...|
|  0.0|(137,[3,9,13,23,26,36,40,42,49,59,62,68,73,77,87,97,103,106,112,124,130],[1.0...|
|  0.0|(137,[3,9,13,23,26,36,40,42,49,59,62,68,72,77,87,97,103,106,112,124,130],[1.0...|
|  0.0|(137,[3,9,13,23,26,36,40,42,49,59,62,67,73,77,87,97,103,106,112,125,130],[1.0...|
|  0.0|(137,[3,9,13,23,26,36,40,42,49,59,62,67,73,77,87,97,103,106,112,124,130],[1.0...|
+-----+--------------------------------------------------------------------------------+
only showing top 5 rows


## Training et Comparaison

On teste 6 familles de modèles disponibles dans `pyspark.ml.classification` :

| Modèle | Idée |
|---|---|
| Régression logistique | modèle linéaire, la référence |
| SVM linéaire (`LinearSVC`) | modèle linéaire, maximise la marge |
| Naive Bayes (Bernoulli) | probabiliste, adapté aux variables binaires |
| Arbre de décision | règles lisibles « si… alors… » |
| Random Forest | moyenne de nombreux arbres |
| Gradient Boosted Trees | arbres entraînés successivement pour corriger les erreurs |
| Perceptron multicouche | petit réseau de neurones |

Pour chaque modèle on mesure : **accuracy**, **F1**, **AUC**, **rappel sur les toxiques** et surtout le **nombre de faux négatifs**.

In [28]:
from pyspark.ml.classification import (
    LogisticRegression, LinearSVC, NaiveBayes, DecisionTreeClassifier,
    RandomForestClassifier, GBTClassifier, MultilayerPerceptronClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator


In [29]:
# Métriques
acc_eval = MulticlassClassificationEvaluator(metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(metricName="f1")
auc_eval = BinaryClassificationEvaluator(metricName="areaUnderROC", rawPredictionCol="rawPrediction")


In [30]:
def evaluate(name, predictions, train_seconds):
    """Calcule les métriques et la matrice de confusion d'un jeu de prédictions."""
    cm = {(r["label"], r["prediction"]): r["count"]
          for r in predictions.groupBy("label", "prediction").count().collect()}
    tp = cm.get((1.0, 1.0), 0)   # toxique prédit toxique
    fn = cm.get((1.0, 0.0), 0)   # toxique prédit comestible  -> DANGEREUX
    fp = cm.get((0.0, 1.0), 0)   # comestible prédit toxique  -> simple gâchis
    return Row(
        modele=name,
        accuracy=round(acc_eval.evaluate(predictions), 4),
        f1=round(f1_eval.evaluate(predictions), 4),
        auc=round(auc_eval.evaluate(predictions), 4),
        rappel_toxique=round(tp / (tp + fn), 4) if tp + fn else None,
        faux_negatifs=fn,
        faux_positifs=fp,
        temps_s=round(train_seconds, 1),
    )

def get_models(n_features):
    return {
        "Régression logistique": LogisticRegression(maxIter=100),
        "SVM linéaire": LinearSVC(maxIter=100),
        "Naive Bayes": NaiveBayes(modelType="bernoulli"),
        "Arbre de décision": DecisionTreeClassifier(maxDepth=5, seed=42),
        "Random Forest": RandomForestClassifier(numTrees=100, maxDepth=5, seed=42),
        "GBT": GBTClassifier(maxIter=50, maxDepth=5, seed=42),
        "Perceptron multicouche": MultilayerPerceptronClassifier(
            layers=[n_features, 16, 2], maxIter=200, seed=42),
    }


def compare_models(train, test, n_features):
    results, fitted = [], {}
    for name, model in get_models(n_features).items():
        start = time.time()
        fitted[name] = model.fit(train)
        elapsed = time.time() - start
        results.append(evaluate(name, fitted[name].transform(test), elapsed))
        print(f"✓ {name} ({elapsed:.1f} s)")
    table = spark.createDataFrame(results).orderBy(F.col("faux_negatifs"), F.col("accuracy").desc())
    return table, fitted


In [31]:
results_df, fitted_models = compare_models(train_prep, test_prep, n_features)
results_df.show(truncate=False)

✓ Régression logistique (2.3 s)
✓ SVM linéaire (3.4 s)
✓ Naive Bayes (0.3 s)
✓ Arbre de décision (0.5 s)
✓ Random Forest (1.7 s)
✓ GBT (12.5 s)
✓ Perceptron multicouche (1.7 s)
+----------------------+--------+------+------+--------------+-------------+-------------+-------+
|modele                |accuracy|f1    |auc   |rappel_toxique|faux_negatifs|faux_positifs|temps_s|
+----------------------+--------+------+------+--------------+-------------+-------------+-------+
|Régression logistique |1.0     |1.0   |1.0   |1.0           |0            |0            |2.3    |
|SVM linéaire          |1.0     |1.0   |1.0   |1.0           |0            |0            |3.4    |
|GBT                   |1.0     |1.0   |1.0   |1.0           |0            |0            |12.5   |
|Perceptron multicouche|1.0     |1.0   |1.0   |1.0           |0            |0            |1.7    |
|Arbre de décision     |0.9929  |0.9929|0.989 |0.9948        |4            |7            |0.5    |
|Random Forest         |0.9865 

**Comment lire ce tableau ?**

- Le tableau est trié par **faux négatifs croissants**, puis par accuracy : le modèle le plus sûr arrive en premier.
- Sur ce dataset, plusieurs modèles atteignent (ou frôlent) **100 %**. C'est connu : les variables comme l'odeur rendent le problème presque linéairement séparable.
- Les modèles **linéaires** (logistique, SVM), le **GBT** et le **perceptron** s'en sortent parfaitement : le one-hot encoding rend le problème quasi linéaire.
- Les **arbres limités en profondeur** (arbre, Random Forest à `maxDepth=5`) laissent passer des toxiques : ils n'ont pas assez de profondeur pour capter toutes les combinaisons. Le Random Forest est même pire que l'arbre seul ici, car chaque arbre ne voit qu'une partie des variables.
- **Naive Bayes** est en retrait : il suppose les variables indépendantes, ce qui est faux ici (ex. `stalk_surface_above_ring` et `stalk_surface_below_ring` sont très corrélées).

Un seul découpage train/test peut être chanceux : on confirme avec la validation croisée.

## Cross-validation

In [32]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [33]:
prep_stages = build_prep_stages(feature_cols)
candidates = ["Régression logistique", "SVM linéaire", "GBT", "Arbre de décision"]

In [34]:
cv_rows = []
for name in candidates:
    model = get_models(n_features)[name]
    cv = CrossValidator(
        estimator=Pipeline(stages=prep_stages + [model]),
        estimatorParamMaps=ParamGridBuilder().build(),   # pas de grille : on mesure juste la stabilité
        evaluator=acc_eval,
        numFolds=5,
        seed=42,
        parallelism=2,
    )
    cv_model = cv.fit(train_df)
    # avgMetrics = moyenne sur les 5 plis ; stdMetrics = écart-type
    # float() : ce sont des numpy.float64, que Spark ne sait pas typer seul
    cv_rows.append(Row(modele=name,
                       accuracy_moyenne=round(float(cv_model.avgMetrics[0]), 4),
                       ecart_type=round(float(cv_model.stdMetrics[0]), 4)))
    print(f"✓ {name}")

spark.createDataFrame(cv_rows).orderBy(F.col("accuracy_moyenne").desc()).show(truncate=False)

✓ Régression logistique
✓ SVM linéaire
✓ GBT
✓ Arbre de décision
+---------------------+----------------+----------+
|modele               |accuracy_moyenne|ecart_type|
+---------------------+----------------+----------+
|Régression logistique|0.9997          |6.0E-4    |
|SVM linéaire         |0.9997          |6.0E-4    |
|GBT                  |0.9994          |0.0013    |
|Arbre de décision    |0.9955          |0.0032    |
+---------------------+----------------+----------+



# Optimiser un modèle qui n'est pas à 100%

L'arbre de décision a l'avantage d'être **lisible** (on peut en tirer des règles), mais à `maxDepth=5` il laisse passer des toxiques.
On cherche avec une **grille d'hyperparamètres** la combinaison qui donne la meilleure accuracy en validation croisée.

- `maxDepth` : profondeur maximale de l'arbre ;
- `minInstancesPerNode` : nombre minimum d'exemples par feuille (limite le sur-apprentissage).

In [35]:
dt = DecisionTreeClassifier(seed=42)

grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [3, 5, 7, 10])
    .addGrid(dt.minInstancesPerNode, [1, 5])
    .build()
)

cv_dt = CrossValidator(
    estimator=Pipeline(stages=build_prep_stages(feature_cols) + [dt]),
    estimatorParamMaps=grid,
    evaluator=acc_eval,
    numFolds=5,
    seed=42,
    parallelism=4,
)
cv_dt_model = cv_dt.fit(train_df)

# Résultat de chaque combinaison testée
for params, score in sorted(zip(grid, cv_dt_model.avgMetrics), key=lambda x: -x[1]):
    p = {k.name: v for k, v in params.items()}
    print(f"maxDepth={p['maxDepth']:>2}  minInstancesPerNode={p['minInstancesPerNode']}  ->  accuracy CV = {score:.4f}")

maxDepth=10  minInstancesPerNode=1  ->  accuracy CV = 0.9997
maxDepth= 7  minInstancesPerNode=5  ->  accuracy CV = 0.9993
maxDepth=10  minInstancesPerNode=5  ->  accuracy CV = 0.9993
maxDepth= 7  minInstancesPerNode=1  ->  accuracy CV = 0.9992
maxDepth= 5  minInstancesPerNode=1  ->  accuracy CV = 0.9955
maxDepth= 5  minInstancesPerNode=5  ->  accuracy CV = 0.9954
maxDepth= 3  minInstancesPerNode=1  ->  accuracy CV = 0.9855
maxDepth= 3  minInstancesPerNode=5  ->  accuracy CV = 0.9855


In [36]:
best_pipeline = cv_dt_model.bestModel
best_tree = best_pipeline.stages[-1]
print("Meilleurs paramètres :", best_tree.getMaxDepth(), "de profondeur,",
      best_tree.getMinInstancesPerNode(), "exemple(s) min. par feuille,", best_tree.numNodes, "nœuds")

best_preds = best_pipeline.transform(test_df)
spark.createDataFrame([evaluate("Arbre optimisé", best_preds, 0)]).show(truncate=False)

Meilleurs paramètres : 10 de profondeur, 1 exemple(s) min. par feuille, 33 nœuds
+--------------+--------+---+---+--------------+-------------+-------------+-------+
|modele        |accuracy|f1 |auc|rappel_toxique|faux_negatifs|faux_positifs|temps_s|
+--------------+--------+---+---+--------------+-------------+-------------+-------+
|Arbre optimisé|1.0     |1.0|1.0|1.0           |0            |0            |0      |
+--------------+--------+---+---+--------------+-------------+-------------+-------+



L'arbre a besoin d'une profondeur d'environ 10, mais reste petit (une trentaine de nœuds) : il atteint 100 % sur le test tout en restant lisible.
`maxDepth=7` fait presque aussi bien en validation croisée

À performance égale avec les modèles de l'étape 6, **on préfère le plus simple et le plus explicable**.
Pour un usage « sécurité », un arbre qui donne des règles vérifiables par un mycologue vaut mieux qu'une boîte noire.

## Les variables qui comptent

Les modèles à base d'arbres donnent une **importance** par colonne du vecteur `features`.
Avec le one-hot encoding, chaque colonne correspond à une modalité (ex. `odor_ohe_n` = « odeur : aucune »).
On récupère ces noms dans les métadonnées du vecteur, puis on regroupe par variable d'origine.

In [37]:
def feature_names(prepared_df, col="features"):
    """Noms des colonnes du vecteur, tirés des métadonnées posées par VectorAssembler."""
    attrs = prepared_df.schema[col].metadata["ml_attr"]["attrs"]
    names = sorted((a["idx"], a["name"]) for group in attrs.values() for a in group)
    return [n for _, n in names]


names = feature_names(best_pipeline.transform(train_df))
importances = best_tree.featureImportances.toArray()

# Importance par modalité
top = sorted(zip(names, importances), key=lambda x: -x[1])[:10]
print("Top 10 des modalités :")
for n, imp in top:
    print(f"  {n:40s} {imp:.3f}")

# Importance cumulée par variable d'origine
by_var = {}
for n, imp in zip(names, importances):
    var = n.split("_ohe")[0]
    by_var[var] = by_var.get(var, 0) + imp
print("\nImportance par variable :")
for var, imp in sorted(by_var.items(), key=lambda x: -x[1]):
    if imp > 0:
        print(f"  {var:28s} {imp:.3f}")

Top 10 des modalités :
  odor_ohe_n                               0.621
  stalk_root_ohe_c                         0.179
  stalk_root_ohe_r                         0.085
  spore_print_color_ohe_r                  0.030
  spore_print_color_ohe_u                  0.024
  bruises_ohe_f                            0.020
  stalk_color_below_ring_ohe_y             0.013
  odor_ohe_l                               0.010
  odor_ohe_a                               0.010
  cap_color_ohe_y                          0.002

Importance par variable :
  odor                         0.641
  stalk_root                   0.263
  spore_print_color            0.054
  bruises                      0.020
  stalk_color_below_ring       0.013
  cap_color                    0.002
  cap_surface                  0.002
  gill_size                    0.002
  stalk_surface_below_ring     0.001
  cap_shape                    0.001


On peut aussi **lire l'arbre** sous forme de règles. Spark affiche les variables par leur numéro (`feature 23`),
on les remplace par leur nom pour que ce soit lisible.

In [38]:
import re

rules = best_tree.toDebugString
rules = re.sub(r"feature (\d+)", lambda m: names[int(m.group(1))], rules)
print(rules)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_b1bf76bb8611, depth=9, numNodes=33, numClasses=2, numFeatures=137
  If (odor_ohe_n in {1.0})
   If (spore_print_color_ohe_r in {0.0})
    If (stalk_color_below_ring_ohe_y in {0.0})
     If (cap_surface_ohe_g in {1.0})
      Predict: 1.0
     Else (cap_surface_ohe_g not in {1.0})
      If (cap_color_ohe_y in {1.0})
       Predict: 1.0
      Else (cap_color_ohe_y not in {1.0})
       If (stalk_surface_below_ring_ohe_y in {0.0})
        If (cap_shape_ohe_c in {1.0})
         Predict: 1.0
        Else (cap_shape_ohe_c not in {1.0})
         If (gill_size_ohe_b in {1.0})
          Predict: 0.0
         Else (gill_size_ohe_b not in {1.0})
          If (bruises_ohe_f in {1.0})
           Predict: 0.0
          Else (bruises_ohe_f not in {1.0})
           Predict: 1.0
       Else (stalk_surface_below_ring_ohe_y not in {0.0})
        If (gill_size_ohe_b in {1.0})
         Predict: 0.0
        Else (gill_size_ohe_b not in {1.0})
       

Lecture :
- `If (odor_ohe_n in {1.0})` signifie « si l'odeur **est** *aucune* » ; `in {0.0}` signifie l'inverse ;
- `Predict: 1.0` = toxique, `Predict: 0.0` = comestible.

On retrouve l'odeur à la racine. Les importances dépendent du modèle : un autre arbre ou un Random Forest pourrait répartir
le poids différemment entre variables corrélées (`stalk_root`, `spore_print_color`…).

## Scénario ou nous n'avons pas les variables discriminantes, juste à l'oeil

In [39]:
field_cols = ["cap_shape", "cap_surface", "cap_color", "bruises", "population", "habitat"]

field_prep = Pipeline(stages=build_prep_stages(field_cols)).fit(train_df)
field_train = field_prep.transform(train_df).select("label", "features").cache()
field_test = field_prep.transform(test_df).select("label", "features").cache()
field_n_features = field_train.first()["features"].size

field_results, field_models = compare_models(field_train, field_test, field_n_features)
field_results.show(truncate=False)

✓ Régression logistique (0.8 s)
✓ SVM linéaire (3.0 s)
✓ Naive Bayes (0.2 s)
✓ Arbre de décision (0.2 s)
✓ Random Forest (1.2 s)
✓ GBT (10.1 s)
✓ Perceptron multicouche (3.1 s)
+----------------------+--------+------+------+--------------+-------------+-------------+-------+
|modele                |accuracy|f1    |auc   |rappel_toxique|faux_negatifs|faux_positifs|temps_s|
+----------------------+--------+------+------+--------------+-------------+-------------+-------+
|GBT                   |0.9858  |0.9858|0.9991|0.9921        |6            |16           |10.1   |
|Perceptron multicouche|0.9858  |0.9858|0.9992|0.9921        |6            |16           |3.1    |
|Random Forest         |0.9536  |0.9536|0.9877|0.9528        |36           |36           |1.2    |
|Arbre de décision     |0.9581  |0.9581|0.9659|0.9502        |38           |27           |0.2    |
|SVM linéaire          |0.9329  |0.9329|0.9259|0.9161        |64           |40           |3.0    |
|Régression logistique |0.8904 

On relance la même comparaison avec les features visibles à l'oeil :
`cap_shape`, `cap_surface`, `cap_color`, `bruises`, `population`, `habitat`.

Là, les modèles se différencient nettement :
- les modèles linéaires et Naive Bayes chutent, car ils ne savent pas combiner les variables entre elles ;
- le **GBT** et le **perceptron multicouche** captent les interactions (« chapeau de telle forme **et** tel habitat ») et restent autour de 98,5 % ;
- l'arbre seul et le Random Forest sont entre les deux.

**Mais** même le meilleur laisse passer des champignons toxiques : en conditions réelles, un tel modèle ne doit **jamais** servir seul à décider de manger un champignon.

### Réduire les faux négatifs en déplaçant le seuil

Par défaut, un modèle prédit « toxique » si la probabilité dépasse 0,5. Pour la sécurité, on peut **abaisser ce seuil** :
on accepte plus de faux positifs (champignons comestibles jetés) pour réduire les faux négatifs.

In [40]:
from pyspark.ml.functions import vector_to_array

gbt_field = field_models["GBT"]
proba = (
    gbt_field.transform(field_test)
    .withColumn("p_toxique", vector_to_array("probability")[1])
    .select("label", "p_toxique")
    .cache()
)

for seuil in [0.5, 0.4, 0.3, 0.2, 0.1]:
    pred = proba.withColumn("prediction", (F.col("p_toxique") >= seuil).cast("double"))
    fn = pred.filter("label = 1 and prediction = 0").count()
    fp = pred.filter("label = 0 and prediction = 1").count()
    print(f"seuil {seuil:.1f} -> faux négatifs : {fn:4d} | faux positifs : {fp:4d}")

seuil 0.5 -> faux négatifs :    6 | faux positifs :   16
seuil 0.4 -> faux négatifs :    3 | faux positifs :   17
seuil 0.3 -> faux négatifs :    0 | faux positifs :   26
seuil 0.2 -> faux négatifs :    0 | faux positifs :   26
seuil 0.1 -> faux négatifs :    0 | faux positifs :   54


C'est le compromis classique **rappel / précision** : à toi de choisir le seuil selon le coût de chaque erreur.
Ici, une erreur de type « toxique prédit comestible » est bien plus grave qu'un champignon comestible jeté.

## Conclusion

| Question | Réponse |
|---|---|
| Quel modèle choisir avec toutes les variables ? | Plusieurs atteignent 100 %. À performance égale, on prend le plus **simple et explicable** : l'arbre optimisé, ou la régression logistique. |
| Quelle variable compte le plus ? | **L'odeur**, de très loin, puis la couleur de sporée. |
| Quel modèle quand le problème est plus dur ? | Le **GBT** (ou le perceptron), qui capte les interactions entre variables. Avec un seuil abaissé à 0,3, plus aucun faux négatif sur le test. |
| Quelle métrique surveiller ? | Les **faux négatifs** (toxiques prédits comestibles), pas seulement l'accuracy. |

**Pour aller plus loin**
- Tuner le GBT du scénario terrain (`maxDepth`, `maxIter`, `stepSize`) avec `CrossValidator`.
- Tester `ChiSqSelector` / `UnivariateFeatureSelector` pour garder les modalités les plus utiles.
- Donner aux arbres les index au lieu du one-hot (ils gèrent nativement les catégories).

In [42]:
spark.stop()